<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/content/dam/news/images/noticies/2016/202-nova-marca-uoc.jpg" align="left" width="45%">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.878 · Trabajo de Fin de Máster · <i>Random Forest</i></p>
<p style="margin: 0; text-align:right;">2025-2 · Máster universitario en Ciencia de datos</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Marcos Rodríguez Soler</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Preprocesamiento de los datos para el modelo _Random Forest_

En este _notebook_ se implementa el _pipeline_ de entrenamiento del modelo **Random Forest** tanto para el conjunto de datos entero como para los distintos _clusters_ del _dataset_. En ambos casos el flujo de trabajo es equivalente y empieza por un procedimiento de ingeniería de características para la posterior creación de los conjuntos de entrenamiento y de prueba, donde es necesario agrupar los datos por producto para garantizar que en los conjuntos de aprendizaje y de prueba se incorporan el mismo porcentaje de observaciones de cada serie temporal de cada artículo. Esta decisión es fundamental debido a que el _dataset_ está compuesto por múltiples series temporales independientes, una por cada producto. Si no se realizase esta partición de forma individual, el conjunto de prueba podría contener series completas de determinados artículos que no habrían sido vistas durante el entrenamiento, obligando al modelo a realizar predicciones sin disponer de información previa sobre su comportamiento histórico. De este modo, al preservar la estructura temporal de cada ítem, el modelo puede aprender los patrones de demanda específicos de cada serie y generalizar adecuadamente en el conjunto de prueba. El objetivo final es que el modelo sea capaz de capturar la dinámica temporal de la demanda, identificando regularidades y patrones comunes que permitan realizar predicciones precisas. Dicho esto, el conjunto de entrenamiento se crea a partir del primer 80% de registros de cada serie temporal, y el conjunto de prueba se confecciona con el 20% restante de cada serie de la demanda. Adicionalmente, en este caso no se ha definido un conjunto de validación, ya que los algoritmos de tipo _bagging_ presentan una menor tendencia al sobreajuste en comparación con otros modelos.

Acto seguido se realiza una búsqueda aleatoria en el algoritmo _Random Forest_ para hallar la combinación de hiperparámetros que minimizan el RMSE del conjunto de entrenamiento en las distintas iteraciones de la validación cruzada. Asimismo, se ha optado por emplear la clase _RandomizedSearchCV_ en lugar de _GridSearchCV_ para reducir el coste computacional de esta operación de optimización. Por otro lado, cada modelo _Random Forest_ se entrena empleando la suma de los errores cuadrados como métrica para partir los nodos, ya que es computacionalmente sencilla de minimizar. Dicho esto, a continuación se listan los hiperparámetros que se han optimizado en la búsqueda aleatoria.

<ul>
    <li><i>n_estimators</i> &#8594; Número de árboles de decisión individuales que conforman el bosque</li>
    <li><i>max_depth</i> &#8594; Profundidad máxima a la que puede llegar un árbol de decisión del bosque</li>
    <li><i>max_features</i> &#8594; Número máximo de variables que se pueden considerar para partir un nodo hoja en un árbol</li>
    <li><i>min_samples_split</i> &#8594; Número mínimo de registros para partir un nodo interno de un árbol</li>
    <li><i>min_samples_leaf</i> &#8594; Cantidad mínima de muestras que deben tener los nodos resultantes de la partición de un nodo</li>
</ul>

Asimismo, para la validación cruzada de la búsqueda aleatoria se emplea la clase _TimeSeriesSplit_, que permite evitar la fuga de información entre el pasado y el futuro. No obstante, el hecho de tener múltiples series temporales en el conjunto de entrenamiento provoca que la validación cruzada con _TimeSeriesSplit_ no sea estrictamente secuencial, pero se considera que este hecho no introduce un sesgo significativo ya que simplemente se está tratando de encontrar la mejor combinación de hiperparámetros del modelo.

Por último, cabe destacar que el análisis de los resultados de los modelos del _notebook_ se realizará en la memoria escrita del trabajo.

<ol style="list-style: none; padding-left: 0;">
    <li>1. <a href="#ej1">Modelo global</a></li>
    <li>2. <a href="#ej2">Modelos por <i>cluster</i></a> <br>
        &nbsp;&nbsp;2.1. <a href="#ej2.1"><i>Cluster top_ventas</i></a> <br>
        &nbsp;&nbsp;2.2. <a href="#ej2.2"><i>Cluster residual</i></a> <br>
        &nbsp;&nbsp;2.3. <a href="#ej2.3"><i>Cluster alta_rotacion</i></a> <br>
        &nbsp;&nbsp;2.4. <a href="#ej2.4"><i>Cluster estandar</i></a> <br>
    </li>
</ol>

In [ ]:
import time
import random

import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

import matplotlib
import matplotlib.pyplot as plt

import statistics

from typing import Any, Dict, List

pd.set_option("display.max_columns", None)
%matplotlib inline

<br><br>Antes de empezar, se define la función _**secuenciador()**_, que sirve para calcular e incorporar nuevas variables derivadas de un conjunto de datos para capturar la dinámica temporal de las distintas series de demanda, con el objetivo de mejorar la capacidad predictiva de los modelos. Esta función devuelve un seguido de nuevas variables que se listan a continuación.

<ul>
    <li><i>lag_i</i> &#8594; Contiene la demanda de un producto <i>i</i> días en el pasado</li>
    <li><i>hor_j</i> &#8594; Indica la demanda de un artículo <i>j</i> días en el futuro</li>
    <li><i>media_rolling_i</i> &#8594; Contiene el promedio de la demanda de un ítem durante los últimos <i>i</i> días</li>
    <li><i>std_rolling_i</i> &#8594; Representa la desviación estándar de la demanda de un producto a lo largo de los últimos <i>i</i> días</li>
    <li><i>max_rolling_i</i> &#8594; Presenta el máximo valor de la demanda de un artículo durante los últimos <i>i</i> días</li>
</ul>

El objetivo de estas variables es captar el carácter temporal de las series para poder anticiparse a picos de demanda o a periodos sin ventas. Además, las variables _hor_j_ son el equivalente a _udsVenta_ en el futuro, lo cual resulta imprescidible en esta problemática ya que, en un contexto operativo, la demanda se debe predecir tantos días en el futuro como dure el ciclo de reposición de cada producto. Sin embargo, no todos los artículos presentan la misma duración de su ciclo de reposición, tal y como se observó durante el **Análisis Exploratorio de los Datos**, por lo que idealmente deberían calcularse tantos horizontes como días dure el ciclo de reposición de cada ítem. No obstante, para obtener un conjunto de datos homogéneo, se opta por predecir tantos días a futuro para todos los productos como dure el ciclo de reposición más largo del _dataset_, que se alarga hasta 56 días. Asimismo, a pesar de que el horizonte de predicción se extiende considerablemente, no se considera necesario aumentar el número de retardos para el cálculo de las variables de la ventana móvil hasta dicho valor, ya que se considera que la información más relevante se encuentra en el comportamiento reciente. Por este motivo, se empleará una ventana de 28 días en el pasado.

In [ ]:
# Función para crear las secuencias de un producto
def secuenciador(df: pd.DataFrame, lag: int, horizonte: int, target: str):
    """Se extraen los últimos valores de las ventas así como las ventas futuras y se calcula la media de ventas en
    el pasado así como su desviación estándar
    Argumentos:
        df: pd.DataFrame -> Conjunto de datos que se desea secuenciar
        lag: int -> Número de periodos pasados que se desean emplear
        horizonte: int -> Número de periodos futuros que se desean emplear
        target: str -> Nombre de la variable objetivo

    Devuelve:
        pd.DataFrame -> DataFrame con la serie secuenciada
    """
    x, y, filas = [], [], []
    for i in range(len(df) - lag - horizonte + 1):
        subventana: pd.DataFrame = df.iloc[i : i + lag]
        x.append(subventana[target].tolist())
          
        siguientes: pd.DataFrame = df.iloc[i + lag : i + lag + horizonte]
        y.append(siguientes[target].tolist())

        filas.append(df.iloc[i + lag])

    diccionario_df = {}
    for i in range(1, lag + 1):
        diccionario_df[f"lag_{i}"] = [x_lag[lag - i] for x_lag in x]

    for i in range(1, horizonte + 1):
        diccionario_df[f"hor_{i}"] = [y_hor[i - 1] for y_hor in y]

    for i in range(1, lag + 1):
        diccionario_df[f"media_rolling_{i}"] = [statistics.mean(x_lag[lag - i :]) for x_lag in x]

    for i in range(2, lag + 1):
        diccionario_df[f"std_rolling_{i}"] = [statistics.stdev(x_lag[lag - i :]) for x_lag in x]

    for i in range(1, lag + 1):
        diccionario_df[f"max_rolling_{i}"] = [max(x_lag[lag - i :]) for x_lag in x]
            
    df_lags_hor: pd.DataFrame = pd.DataFrame(diccionario_df)
    df_variables: pd.DataFrame = pd.DataFrame(filas).reset_index(drop=True)

    return pd.concat([df_lags_hor, df_variables], axis=1)

<br><br>
A continuación se definen las funciones _**rmse()**_ y _**rmsse_producto()**_, que permiten evaluar el error de las predicciones de los modelos.

La función _**rmse()**_ calcula la raíz del error cuadrático medio entre la demanda real y la demanda predicha sobre el conjunto de prueba. Por otro lado, la función _**rmsse_producto()**_ calcula el error cuadrático medio escalado cada producto y un horizonte temporal concreto, normalizando el error respecto a la variabilidad histórica observada en el conjunto de entrenamiento de cada serie. Este enfoque permite comparar el rendimiento de los modelos entre productos con distintos niveles de demanda.

Para la evaluación de los modelos, ambas métricas se calculan inicialmente para cada combinación de producto, $p$, y horizonte temporal, $h$. Posteriormente, para cada producto se promedian únicamente los errores correspondientes a los horizontes comprendidos dentro del ciclo de aprovisionamiento de cada producto, definido como la suma del tiempo de entrega, $L$, y los días entre pedidos, $R$. 

$error_p = \frac{1}{R+L} \sum_{h=1}^{R+L} error_{p,h}$

Finalmente, el rendimiento global de cada modelo se obtiene promediando los errores operativos de todos los productos, $N$.

$error_{modelo} = \frac{1}{N} \sum_{p=1}^{N} error_p$

En el caso del RMSE, esta métrica se utilizará posteriormente en el apartado **Evaluación del Impacto Económico** durante la simulación de inventario para aproximar la incertidumbre asociada a las predicciones y analizar su efecto sobre las decisiones de reposición y los costes operativos del sistema.

In [ ]:
# Funciones para calcular el RMSE y el RMSSE
def rmse(y_real: pd.Series, y_pred: pd.Series) -> float:
    """Devuelve el RMSE de las predicciones de un modelo sobre un conjunto de prueba

    Argumentos:
        y_real: pd.Series -> Demanda real del conjunto de prueba
        y_pred: pd.Series -> Demanda predicha para el conjunto de prueba

    Devuelve
        float -> RMSE
    """
    return np.sqrt(np.mean(np.abs(y_real - y_pred) ** 2))


def rmsse_producto(df: pd.DataFrame, df_test: pd.DataFrame, horizonte: int) -> Dict[str, float]:
    """Devuelve el RMSSE de cada producto para un horizonte temporal concreto

    Argumentos:
        df: pd.DataFrame -> DataFrame completo del conjunto de datos
        df_test: pd.DataFrame -> Resultados del conjunto de prueba
        horizonte: int -> Horizonte temporal

    Devuelve:
        Dict[str, float] -> RMSSE de cada producto en un horizonte temporal concreto
    """
    valores: Dict[str, float] = {}

    for producto, grupo_test in df_test.groupby("producto"):

        y_real: np.ndarray = grupo_test[f"real_{horizonte}"].values
        y_pred: np.ndarray = grupo_test[f"pred_{horizonte}"].values

        numerador: float = np.mean((y_real - y_pred) ** 2)

        # Datos de entrenamiento del producto
        grupo_train: pd.DataFrame = (
            df[df["producto"] == producto]
            .sort_values("fecha")
        )

        PARTICION: int = int(len(grupo_train) * 0.8)

        y_train: np.ndarray = (
            grupo_train[f"hor_{horizonte}"]
            .iloc[:PARTICION]
            .values
        )

        denominador: float = np.mean(
            (y_train[1:] - y_train[:-1]) ** 2
        )

        if denominador != 0 and not np.isnan(denominador):
            valores[producto] = np.sqrt(numerador / denominador)

    return valores

<br><br><a id="ej1"></a>
# 1. Modelo global

En esta sección se lleva a cabo el entrenamiento del modelo _Random Forest_ que emplea todos los productos del conjunto de datos. Inicialmente, se importan los datos resultantes del **Análisis Exploratorio de los Datos** y se calculan las variables de ventana móvil hasta 28 días en el pasado para cada muestra, así como los distintos horizontes de la demanda.

In [ ]:
# Se importan los datos preprocesados del análisis exploratorio de datos
ruta_dataset: str = "../AED/dataset_preprocesado.csv"
dataset_global: pd.DataFrame = pd.read_csv(ruta_dataset).drop("Unnamed: 0", axis=1)

In [ ]:
# Se crean las nuevas variables para el conjunto de datos completo para entrenar un modelo global
LAG: int = 28
HORIZONTE: int = max(dataset_global["diasLeadtime"] + dataset_global["diasEntrePedidos"])
dataset_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in dataset_global.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    dataset_secuenciado.append(df_producto_sec)

dataset_secuenciado: pd.DataFrame = pd.concat(dataset_secuenciado, ignore_index=True)

<br><br>De todo el conjunto de variables adicionales que se han calculado, se seleccionan únicamente aquellas cada 7 días en el pasado, ya que este periodo de tiempo se corresponde con una semana. Esta decisión se debe a que cada una de estas variables captura los distintos patrones de la demanda a lo largo de una semana en el pasado, permitiendo al modelo aprender el factor temporal de los datos con variables lo suficientemente espaciadas en el tiempo. Asimismo, seleccionar más variables de este tipo introduciría demasiada multicolinealidad, ya que muchos de estos atributos están fuertemente correlacionados entre sí. De esta forma, durante el modelado resulta más sencillo aislar el efecto individual de cada predictor sobre la variable dependiente, lo que permite comprender más fácilmente el papel específico de cada característica en la toma de decisiones del modelo. En cuanto a las variables originales del _dataset_, se seleccionan todas las que se han analizado durante el **Análisis Exploratorio de los Datos**, ya que aunque muchas no presentaban una fuerte correlación con la variable objetivo, se considera que pueden existir correlaciones no lineales que la matriz de correlaciones no es capaz de captar pero que sin embargo los modelos de aprendizaje automático sí pueden detectar.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos global
dataset_global_subset: pd.DataFrame = (
    dataset_secuenciado[
        [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles", "dia_semana_str_Jueves",
            "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo"
        ] + [f"mes_str_{i}" for i in range(1, 13)] + [
            "lag_7", "lag_14", "lag_21", "lag_28", "media_rolling_7", "media_rolling_14", "media_rolling_21", "media_rolling_28",
            "max_rolling_7", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7", "std_rolling_14", "std_rolling_21",
            "std_rolling_28", "udsVenta"
        ] 
    ]
)

In [ ]:
# Matriz de correlaciones con las nuevas variables explicativas
corr: pd.DataFrame = dataset_global_subset.drop(columns=["producto", "fecha", "udsVenta"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones del conjunto de datos global tras realizar un procedimiento de ingeniería de características")

plt.tight_layout()
plt.show()

La matriz de correlaciones demuestra que las variables derivadas mediante ventanas móviles presentan una relación significativa con las variables objetivo. En particular, las medias, máximos y desviaciones estándar calculadas sobre distintas ventanas temporales permiten capturar de forma consistente el nivel de demanda, su variabilidad y la presencia de valores extremos en periodos recientes. No obstante, estos atributos también muestran una elevada correlación entre ellos debido a su proximidad en el tiempo.

Dicho esto, se procede con la creación de los conjuntos de entrenamiento y de prueba, así como con la optimización de hiperparámetros y el propio entrenamiento del modelo _Random Forest_ para el conjunto de datos global.

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del conjunto de datos global
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_dataset_global: Dict[str, pd.DataFrame] = {}
lista_x_train: List[pd.DataFrame] = []
lista_x_test: List[pd.DataFrame] = []
lista_y_train: List[pd.DataFrame] = []
lista_y_test: List[pd.DataFrame] = []

for producto, df_producto in dataset_global_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    lista_x_train.append(x_producto.iloc[:PARTICION])
    lista_x_test.append(x_producto.iloc[PARTICION:])
    
    lista_y_train.append(y_producto.iloc[:PARTICION])
    lista_y_test.append(y_producto.iloc[PARTICION:])

conjuntos_dataset_global["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_dataset_global["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_dataset_global["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_dataset_global["y_test"]  = pd.concat(lista_y_test, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_dataset_global["x_train"].shape[0]} filas y {conjuntos_dataset_global["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_dataset_global["y_train"].shape[0]} filas y {conjuntos_dataset_global["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_dataset_global["x_test"].shape[0]} filas y {conjuntos_dataset_global["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_dataset_global["y_test"].shape[0]} filas y {conjuntos_dataset_global["y_test"].shape[1]} columnas")

In [ ]:
# Se inicia la búsqueda aleatoria del modelo Random Forest para el conjunto de datos global
SEMILLA: int = 42
N_ITER: int = 20
tscv: TimeSeriesSplit = TimeSeriesSplit(n_splits=5)

modelo_forest_search: RandomForestRegressor = RandomForestRegressor(
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

param_search_forest: Dict[str, List[Any]] = {
    "n_estimators": [100, 200],
    "max_depth": [5, 7, 10],
    "max_features": ["sqrt", 0.5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [2, 5]
}

search_forest_global: RandomizedSearchCV = RandomizedSearchCV(
    estimator=modelo_forest_search,
    param_distributions=param_search_forest,
    n_iter=N_ITER,
    verbose=1,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=SEMILLA
)

tiempo_inicial: float = time.time()
search_forest_global.fit(conjuntos_dataset_global["x_train"], conjuntos_dataset_global["y_train"])
tiempo_final: float = time.time()

print(
    "La búsqueda aleatoria del modelo Random Forest para el dataset global tardó "
    f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
)

In [ ]:
# Se muestran los valores óptimos de los hiperparámetros del algoritmo tras la búsqueda aleatoria
n_estimators_opt: int = search_forest_global.best_params_["n_estimators"]
max_depth_opt: int = search_forest_global.best_params_["max_depth"]
max_features_opt: int | str = search_forest_global.best_params_["max_features"]
min_samples_split_opt: int = search_forest_global.best_params_["min_samples_split"]
min_samples_leaf_opt: int = search_forest_global.best_params_["min_samples_leaf"]

i = -1
for comb in search_forest_global.cv_results_["params"]:
    i += 1
    if (
        comb["n_estimators"] == n_estimators_opt and 
        comb["max_depth"] == max_depth_opt and 
        comb["max_features"] == max_features_opt and
        comb["min_samples_split"] == min_samples_split_opt and
        comb["min_samples_leaf"] == min_samples_leaf_opt
    ):
        break

print(
    f"El mejor número de árboles es n_estimators={n_estimators_opt}, la profundidad máxima óptima "
    f"de los árboles es de max_depth={max_depth_opt}, el mejor criterio para partir una hoja de "
    f"un árbol según el número de variables a tener en cuenta es max_features={max_features_opt}, "
    f"el número mínimo de muestras de un split debe ser min_samples_split={min_samples_split_opt}, "
    f"y el número mínimo de muestras por hoja debe ser min_samples_leaf={min_samples_leaf_opt}", "\n\n"
)
print(
    f"Con esta combinación de hiperparámetros, durante la validación cruzada para las series temporales "
    f"se obtuvo un valor promedio del RMSE de {-search_forest_global.cv_results_["mean_test_score"][i]:.4f} con "
    f"una desviación estándar de {search_forest_global.cv_results_["std_test_score"][i]:.4f}"
)

In [ ]:
# Se entrena el modelo Random Forest con los mejores hiperparámetros empleando el conjunto de datos global
tiempo_inicial: float = time.time()

modelo_forest_global: RandomForestRegressor = RandomForestRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    max_features=max_features_opt,
    min_samples_split=min_samples_split_opt,
    min_samples_leaf=min_samples_leaf_opt,
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

modelo_forest_global.fit(conjuntos_dataset_global["x_train"], conjuntos_dataset_global["y_train"])

tiempo_final: float = time.time()

print(f"El modelo global tardó {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset global
y_pred_global: np.ndarray = modelo_forest_global.predict(conjuntos_dataset_global["x_test"])

y_pred_global_df: pd.DataFrame = pd.DataFrame(
    y_pred_global,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in dataset_global_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_global_df: pd.DataFrame = pd.DataFrame(
    conjuntos_dataset_global["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_modelo_global: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_global_df, y_pred_global_df],
    axis=1
)

resultados_modelo_global["fecha"] = pd.to_datetime(resultados_modelo_global["fecha"])

In [ ]:
# Se calculan los promedios del RMSE en el conjunto de prueba del dataset global
rmses: Dict[str, float] = {
    i: rmse(
        resultados_modelo_global[f"real_{i}"],
        resultados_modelo_global[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo global")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br> Se calculan los RMSE y RMSSE a nivel de horizonte por producto del modelo global. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del dataset global por producto y se exportan para la evaluación económica
rmses_global: List[pd.DataFrame] = []

for producto, df_producto in resultados_modelo_global.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_global.append(rmses_df)

rmses_global: pd.DataFrame = pd.concat(rmses_global, ignore_index=True)
rmses_global.to_csv("rmses_random_forest_global.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo global
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        dataset_global_subset,
        resultados_modelo_global,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_modelo_global.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo global
rmsse_modelo_global: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo global es {rmsse_modelo_global:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del dataset global
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_modelo_global["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_modelo_global[resultados_modelo_global["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del dataset global")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se visualiza la importancia de cada atributo empleado en el modelo global
plt.figure(figsize=(10, 6))
plt.bar(
    conjuntos_dataset_global["x_train"].columns,
    modelo_forest_global.feature_importances_,
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1
)

plt.title("Importancia de cada atributo del dataset global en las decisiones del algoritmo Random Forest")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importancia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
productos = [4]

for producto in productos:
    fig, ax = plt.subplots(figsize=(8, 7))
    
    ax.bar(
        conjuntos_dataset_global["x_train"].columns,
        modelo_forest_global.feature_importances_,
        color="blue",
        alpha=0.6,
        edgecolor="black",
        linewidth=1
    )
    
    ax.set_title("Importancia de las variables")
    ax.tick_params("x", rotation=45)
    ax.set_ylabel("Importancia")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

    plt.rcParams.update({
        "font.size": 20,
        "axes.titlesize": 21,
        "axes.labelsize": 19,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17
    })
    
    plt.tight_layout()
    plt.savefig(f"GRÁFICOS/importancia_variables.png", dpi=300)
    plt.show()

In [ ]:
# Se exportan los resultados
resultados_modelo_global.to_csv("res_random_forest_global.csv")

<br><br><a id="ej2"></a>
# 2. Modelos por _cluster_

En este apartado se entrena un modelo _Random Forest_ para cada _cluster_ identificado en el _notebook_ de nombre _clustering_.

<a id="ej2.1"></a>
## 2.1. _Cluster top_ventas_

A continuación se entrena el modelo utilizando los datos del _cluster top_ventas_. El flujo de trabajo equivale al del _dataset_ global, por lo que primero se calculan las variables de ventana móvil con la función _**secuenciador()**_. En este caso, el horizonte de la demanda que se calcula para todas las muestras es el más grande de todo el _cluster_, el cual puede no coincidir con el que se obtuvo en el modelo global.

In [ ]:
# Se extrae el grupo top_ventas del dataset global
top_ventas: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "top_ventas"]

In [ ]:
# Se crean las nuevas variables para el cluster top_ventas
HORIZONTE: int = max(top_ventas["diasLeadtime"] + top_ventas["diasEntrePedidos"])
top_ventas_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in top_ventas.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    top_ventas_secuenciado.append(df_producto_sec)

top_ventas_secuenciado: pd.DataFrame = pd.concat(top_ventas_secuenciado, ignore_index=True)

<br><br>De la misma forma que en el caso del conjunto de datos global, se seleccionan únicamente las variables de ventana móvil calculadas cada 7 días en el pasado para capturar la dinámica temporal de la demanda en periodos de varias semanas. También se añaden las demás variables originales del _dataset_.

In [ ]:
# Se selecciona el subconjunto de variables de interés del cluster top_ventas
top_ventas_subset: pd.DataFrame = (
    top_ventas_secuenciado[
        [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles", "dia_semana_str_Jueves",
            "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo"
        ] + [f"mes_str_{i}" for i in range(1, 13)] + [
            "lag_7", "lag_14", "lag_21", "lag_28", "media_rolling_7", "media_rolling_14", "media_rolling_21", "media_rolling_28",
            "max_rolling_7", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7", "std_rolling_14", "std_rolling_21",
            "std_rolling_28", "udsVenta"
        ] 
    ]
)

In [ ]:
# Matriz de correlaciones con las nuevas variables explicativas
corr: pd.DataFrame = top_ventas_subset.drop(columns=["producto", "fecha", "udsVenta"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones de top_ventas tras realizar un procedimiento de ingeniería de características")

plt.tight_layout()
plt.show()

De forma similar al _dataset_ global, la nueva matriz de correlaciones demuestra que las nuevas variables calculadas presentan una fuerte correlación con las variables objetivo, por lo que su inclusión debería mejorar la señal predictiva de los distintos horizontes de la demanda.

Así pues, se procede tanto con la optimización de hiperparámetros como con el entrenamiento del modelo _Random Forest_ para los datos de este _cluster_.

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster top_ventas
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_top_ventas: Dict[str, pd.DataFrame] = {}
lista_x_train: List[pd.DataFrame] = []
lista_x_test: List[pd.DataFrame] = []
lista_y_train: List[pd.DataFrame] = []
lista_y_test: List[pd.DataFrame] = []

for producto, df_producto in top_ventas_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes)
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    lista_x_train.append(x_producto.drop("fecha", axis=1).iloc[:PARTICION])
    lista_x_test.append(x_producto.drop("fecha", axis=1).iloc[PARTICION:])
    
    lista_y_train.append(y_producto.iloc[:PARTICION])
    lista_y_test.append(y_producto.iloc[PARTICION:])

conjuntos_top_ventas["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_top_ventas["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_top_ventas["y_train"] = pd.concat(lista_y_train, ignore_index=True).squeeze()
conjuntos_top_ventas["y_test"] = pd.concat(lista_y_test, ignore_index=True).squeeze()

print(f"El conjunto x_train contiene {conjuntos_top_ventas["x_train"].shape[0]} filas y {conjuntos_top_ventas["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_top_ventas["y_train"].shape[0]} filas y {conjuntos_top_ventas["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_top_ventas["x_test"].shape[0]} filas y {conjuntos_top_ventas["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_top_ventas["y_test"].shape[0]} filas y {conjuntos_top_ventas["y_test"].shape[1]} columnas")

In [ ]:
# Se inicia la búsqueda aleatoria del modelo Random Forest para el conjunto top_ventas
search_forest_top_ventas: RandomizedSearchCV = RandomizedSearchCV(
    estimator=modelo_forest_search,
    param_distributions=param_search_forest,
    n_iter=N_ITER,
    verbose=1,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=SEMILLA
)

tiempo_inicial: float = time.time()
search_forest_top_ventas.fit(conjuntos_top_ventas["x_train"], conjuntos_top_ventas["y_train"])
tiempo_final: float = time.time()

print(
    "La búsqueda aleatoria del modelo Random Forest para el cluster top_ventas tardó "
    f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
)

In [ ]:
# Se muestran los valores óptimos de los hiperparámetros del algoritmo tras la búsqueda aleatoria
n_estimators_opt: int = search_forest_top_ventas.best_params_["n_estimators"]
max_depth_opt: int = search_forest_top_ventas.best_params_["max_depth"]
max_features_opt: int | str = search_forest_top_ventas.best_params_["max_features"]
min_samples_split_opt: int = search_forest_top_ventas.best_params_["min_samples_split"]
min_samples_leaf_opt: int = search_forest_top_ventas.best_params_["min_samples_leaf"]

i = -1
for comb in search_forest_top_ventas.cv_results_["params"]:
    i += 1
    if (
        comb["n_estimators"] == n_estimators_opt and 
        comb["max_depth"] == max_depth_opt and 
        comb["max_features"] == max_features_opt and
        comb["min_samples_split"] == min_samples_split_opt and
        comb["min_samples_leaf"] == min_samples_leaf_opt
    ):
        break

print(
    f"El mejor número de árboles es n_estimators={n_estimators_opt}, la profundidad máxima óptima "
    f"de los árboles es de max_depth={max_depth_opt}, el mejor criterio para partir una hoja de "
    f"un árbol según el número de variables a tener en cuenta es max_features={max_features_opt}, "
    f"el número mínimo de muestras de un split debe ser min_samples_split={min_samples_split_opt}, "
    f"y el número mínimo de muestras por hoja debe ser min_samples_leaf={min_samples_leaf_opt}", "\n\n"
)
print(
    f"Con esta combinación de hiperparámetros, durante la validación cruzada para las series temporales "
    f"se obtuvo un valor promedio del RMSE de {-search_forest_top_ventas.cv_results_["mean_test_score"][i]:.4f} con "
    f"una desviación estándar de {search_forest_top_ventas.cv_results_["std_test_score"][i]:.4f}"
)

In [ ]:
# Se entrena el modelo Random Forest con los mejores hiperparámetros empleando el cluster top_ventas
tiempo_inicial: float = time.time()

modelo_forest_top_ventas: RandomForestRegressor = RandomForestRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    max_features=max_features_opt,
    min_samples_split=min_samples_split_opt,
    min_samples_leaf=min_samples_leaf_opt,
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

modelo_forest_top_ventas.fit(conjuntos_top_ventas["x_train"], conjuntos_top_ventas["y_train"])

tiempo_final: float = time.time()

print(f"El modelo de top_ventas tardó {round(tiempo_final - tiempo_inicial, 4)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del cluster top_ventas
y_pred_top_ventas: np.ndarray = modelo_forest_top_ventas.predict(conjuntos_top_ventas["x_test"])

y_pred_top_ventas_df: pd.DataFrame = pd.DataFrame(
    y_pred_top_ventas,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in top_ventas_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_top_ventas_df: pd.DataFrame = pd.DataFrame(
    conjuntos_top_ventas["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_top_ventas: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_top_ventas_df, y_pred_top_ventas_df],
    axis=1
)

resultados_top_ventas["fecha"] = pd.to_datetime(resultados_top_ventas["fecha"])

In [ ]:
# Se calculan los promedios de los errores cometidos en el conjunto de prueba del cluster top_ventas
rmses: Dict[str, float] = {
    i: rmse(
        resultados_top_ventas[f"real_{i}"],
        resultados_top_ventas[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales de top_ventas")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster top_ventas_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba de top_ventas por producto y se exportan para la evaluación económica
rmses_top_ventas: List[pd.DataFrame] = []

for producto, df_producto in resultados_top_ventas.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_top_ventas.append(rmses_df)

rmses_top_ventas: pd.DataFrame = pd.concat(rmses_top_ventas, ignore_index=True)
rmses_top_ventas.to_csv("rmses_random_forest_top_ventas.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo top_ventas
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        top_ventas_subset,
        resultados_top_ventas,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_top_ventas.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo top_ventas
rmsse_modelo_top_ventas: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo top_ventas es {rmsse_modelo_top_ventas:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios de top_ventas
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_top_ventas["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_top_ventas[resultados_top_ventas["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} de top_ventas")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se visualiza la importancia de cada atributo empleado en el modelo de top_ventas
plt.figure(figsize=(10, 6))
plt.bar(
    conjuntos_top_ventas["x_train"].columns,
    modelo_forest_top_ventas.feature_importances_,
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1
)

plt.title("Importancia de cada atributo del cluster top_ventas en las decisiones del algoritmo Random Forest")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importancia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_top_ventas.to_csv("res_random_forest_top_ventas.csv")

<a id="ej2.2"></a>
## 2.2. _Cluster residual_

En este apartado se utilizan los productos del grupo _residual_ para entrenar un modelo _Random Forest_ siguiendo el mismo flujo de trabajo que en el apartado anterior.

In [ ]:
# Se extrae el cluster residual del dataset global
residual: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "residual"]

In [ ]:
# Se crean las nuevas variables para el cluster residual
HORIZONTE: int = max(residual["diasLeadtime"] + residual["diasEntrePedidos"])
residual_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in residual.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    residual_secuenciado.append(df_producto_sec)

residual_secuenciado: pd.DataFrame = pd.concat(residual_secuenciado, ignore_index=True)

<br><br>De todas las variables que contiene el _cluster residual_, se escogen las del conjunto de datos original junto con las características computadas con la ventana móvil de 28 días en el pasado y los horizontes de la demanda calculados.

In [ ]:
# Se selecciona el subconjunto de variables de interés del cluster residual
residual_subset: pd.DataFrame = (
    residual_secuenciado[
        [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles", "dia_semana_str_Jueves",
            "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo"
        ] + [f"mes_str_{i}" for i in range(1, 13)] + [
            "lag_7", "lag_14", "lag_21", "lag_28", "media_rolling_7", "media_rolling_14", "media_rolling_21", "media_rolling_28",
            "max_rolling_7", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7", "std_rolling_14", "std_rolling_21",
            "std_rolling_28", "udsVenta"
        ] 
    ]
)

In [ ]:
# Matriz de correlaciones con las nuevas variables explicativas
corr: pd.DataFrame = residual_subset.drop(columns=["producto", "fecha", "udsVenta"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones de residual tras realizar un procedimiento de ingeniería de características")

plt.tight_layout()
plt.show()

A diferencia del _cluster top_ventas_, las nuevas variables calculadas presentan una correlación más débil con los horizontes de la demanda. Esto puede deberse a la naturaleza residual del _cluster_, ya que conforma la agrupación de varios grupos más pequños cuyos tamaños no eran lo suficientemente grandes como para poder entrenar un algoritmo de _machine learning_ robusto, y que además presentaban factores característicos distintos entre sí. Esta observación también puede explicar la correlación débil entre los distintos horizontes temporales.

Teniendo esto en cuenta, se entrena el modelo _Random Forest_ con las muestras de este grupo.

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster residual
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_residual: Dict[str, pd.DataFrame] = {}
lista_x_train: List[pd.DataFrame] = []
lista_x_test: List[pd.DataFrame] = []
lista_y_train: List[pd.DataFrame] = []
lista_y_test: List[pd.DataFrame] = []

for producto, df_producto in residual_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes)
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    lista_x_train.append(x_producto.drop("fecha", axis=1).iloc[:PARTICION])
    lista_x_test.append(x_producto.drop("fecha", axis=1).iloc[PARTICION:])
    
    lista_y_train.append(y_producto.iloc[:PARTICION])
    lista_y_test.append(y_producto.iloc[PARTICION:])

conjuntos_residual["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_residual["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_residual["y_train"] = pd.concat(lista_y_train, ignore_index=True).squeeze()
conjuntos_residual["y_test"] = pd.concat(lista_y_test, ignore_index=True).squeeze()

print(f"El conjunto x_train contiene {conjuntos_residual["x_train"].shape[0]} filas y {conjuntos_residual["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_residual["y_train"].shape[0]} filas y {conjuntos_residual["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_residual["x_test"].shape[0]} filas y {conjuntos_residual["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_residual["y_test"].shape[0]} filas y {conjuntos_residual["y_test"].shape[1]} columnas")

In [ ]:
# Se inicia la búsqueda aleatoria del modelo Random Forest para el conjunto residual
search_forest_residual: RandomizedSearchCV = RandomizedSearchCV(
    estimator=modelo_forest_search,
    param_distributions=param_search_forest,
    n_iter=N_ITER,
    verbose=1,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=SEMILLA
)

tiempo_inicial: float = time.time()
search_forest_residual.fit(conjuntos_residual["x_train"], conjuntos_residual["y_train"])
tiempo_final: float = time.time()

print(
    "La búsqueda aleatoria del modelo Random Forest para el cluster residual tardó "
    f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
)

In [ ]:
# Se muestran los valores óptimos de los hiperparámetros del algoritmo tras la búsqueda aleatoria
n_estimators_opt: int = search_forest_residual.best_params_["n_estimators"]
max_depth_opt: int = search_forest_residual.best_params_["max_depth"]
max_features_opt: int | str = search_forest_residual.best_params_["max_features"]
min_samples_split_opt: int = search_forest_residual.best_params_["min_samples_split"]
min_samples_leaf_opt: int = search_forest_residual.best_params_["min_samples_leaf"]

i = -1
for comb in search_forest_residual.cv_results_["params"]:
    i += 1
    if (
        comb["n_estimators"] == n_estimators_opt and 
        comb["max_depth"] == max_depth_opt and 
        comb["max_features"] == max_features_opt and
        comb["min_samples_split"] == min_samples_split_opt and
        comb["min_samples_leaf"] == min_samples_leaf_opt
    ):
        break

print(
    f"El mejor número de árboles es n_estimators={n_estimators_opt}, la profundidad máxima óptima "
    f"de los árboles es de max_depth={max_depth_opt}, el mejor criterio para partir una hoja de "
    f"un árbol según el número de variables a tener en cuenta es max_features={max_features_opt}, "
    f"el número mínimo de muestras de un split debe ser min_samples_split={min_samples_split_opt}, "
    f"y el número mínimo de muestras por hoja debe ser min_samples_leaf={min_samples_leaf_opt}", "\n\n"
)
print(
    f"Con esta combinación de hiperparámetros, durante la validación cruzada para las series temporales "
    f"se obtuvo un valor promedio del RMSE de {-search_forest_residual.cv_results_["mean_test_score"][i]:.4f} con "
    f"una desviación estándar de {search_forest_residual.cv_results_["std_test_score"][i]:.4f}"
)

In [ ]:
# Se entrena el modelo Random Forest con los mejores hiperparámetros empleando el cluster residual
tiempo_inicial: float = time.time()

modelo_forest_residual: RandomForestRegressor = RandomForestRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    max_features=max_features_opt,
    min_samples_split=min_samples_split_opt,
    min_samples_leaf=min_samples_leaf_opt,
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

modelo_forest_residual.fit(conjuntos_residual["x_train"], conjuntos_residual["y_train"])

tiempo_final: float = time.time()

print(f"El modelo del cluster residual tardó {round(tiempo_final - tiempo_inicial, 4)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del cluster residual
y_pred_residual: np.ndarray = modelo_forest_residual.predict(conjuntos_residual["x_test"])

y_pred_residual_df: pd.DataFrame = pd.DataFrame(
    y_pred_residual,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in residual_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_residual_df: pd.DataFrame = pd.DataFrame(
    conjuntos_residual["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_residual: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_residual_df, y_pred_residual_df],
    axis=1
)

resultados_residual["fecha"] = pd.to_datetime(resultados_residual["fecha"])

In [ ]:
# Se calculan los promedios de los errores cometidos en el conjunto de prueba del cluster residual
rmses: Dict[str, float] = {
    i: rmse(
        resultados_residual[f"real_{i}"],
        resultados_residual[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del cluster residual")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster residual_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster residual por producto y se exportan para la evaluación económica
rmses_residual: List[pd.DataFrame] = []

for producto, df_producto in resultados_residual.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_residual.append(rmses_df)

rmses_residual: pd.DataFrame = pd.concat(rmses_residual, ignore_index=True)
rmses_residual.to_csv("rmses_random_forest_residual.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del grupo residual
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        residual_subset,
        resultados_residual,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_residual.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo residual
rmsse_modelo_residual: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo del cluster residual es {rmsse_modelo_residual:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster residual
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_residual["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_residual[resultados_residual["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster residual")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se visualiza la importancia de cada atributo empleado en el modelo del cluster residual
plt.figure(figsize=(10, 6))
plt.bar(
    conjuntos_residual["x_train"].columns,
    modelo_forest_residual.feature_importances_,
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1
)

plt.title("Importancia de cada atributo del cluster residual en las decisiones del algoritmo Random Forest")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importancia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_residual.to_csv("res_random_forest_residual.csv")

<a id="ej2.3"></a>
## 2.3. _Cluster alta_rotacion_

El objetivo de esta sección es entrenar un modelo _Random Forest_ para el grupo _alta_rotacion_. Siguiendo el mismo flujo de trabajo que se ha planteado para cada algoritmo entrenado en este _notebook_, primero se calculan los campos de ventana móvil con la función _**secuenciador()**_.

In [ ]:
# Se extrae el grupo alta_rotacion del dataset global
alta_rotacion: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "alta_rotacion"]

In [ ]:
# Se crean las nuevas variables para el cluster alta_rotacion
HORIZONTE: int = max(alta_rotacion["diasLeadtime"] + alta_rotacion["diasEntrePedidos"])
alta_rotacion_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in alta_rotacion.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    alta_rotacion_secuenciado.append(df_producto_sec)

alta_rotacion_secuenciado: pd.DataFrame = pd.concat(alta_rotacion_secuenciado, ignore_index=True)

<br><br>Para crear los conjuntos de aprendizaje y de prueba del _cluster alta_rotacion_, se seleccionan las variables explicativas del _dataset_ original analizado durante el **Análisis Exploratorio de los Datos**, añadiendo además los atributos calculados con las ventanas móviles cada 7 días en el pasado.

In [ ]:
# Se selecciona el subconjunto de variables de interés del cluster alta_rotacion
alta_rotacion_subset: pd.DataFrame = (
    alta_rotacion_secuenciado[
        [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles", "dia_semana_str_Jueves",
            "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo"
        ] + [f"mes_str_{i}" for i in range(1, 13)] + [
            "lag_7", "lag_14", "lag_21", "lag_28", "media_rolling_7", "media_rolling_14", "media_rolling_21", "media_rolling_28",
            "max_rolling_7", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7", "std_rolling_14", "std_rolling_21",
            "std_rolling_28", "udsVenta"
        ] 
    ]
)

In [ ]:
# Matriz de correlaciones con las nuevas variables explicativas
corr: pd.DataFrame = alta_rotacion_subset.drop(columns=["producto", "fecha", "udsVenta"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones de alta_rotacion tras realizar un procedimiento de ingeniería de características")

plt.tight_layout()
plt.show()

Las variables calculadas sobre distintos días en el pasado presentan correlaciones moderadas con la demanda. Este comportamiento es bastante parecido al del grupo _top_ventas_, por lo que se concluye que la demanda futura sigue dependiendo en gran medida del comportamiento reciente de la serie. Asimismo, se continua observando una fuerte multicolinealidad entre estas variables, derivada de la naturaleza acumulativa de las ventanas temporales.

Dicho esto, continuadamente se entrena el modelo _Random Forest_ para el _cluster alta_rotacion_.

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster alta_rotacion
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_alta_rotacion: Dict[str, pd.DataFrame] = {}
lista_x_train: List[pd.DataFrame] = []
lista_x_test: List[pd.DataFrame] = []
lista_y_train: List[pd.DataFrame] = []
lista_y_test: List[pd.DataFrame] = []

for producto, df_producto in alta_rotacion_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes)
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    lista_x_train.append(x_producto.drop("fecha", axis=1).iloc[:PARTICION])
    lista_x_test.append(x_producto.drop("fecha", axis=1).iloc[PARTICION:])
    
    lista_y_train.append(y_producto.iloc[:PARTICION])
    lista_y_test.append(y_producto.iloc[PARTICION:])

conjuntos_alta_rotacion["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_alta_rotacion["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_alta_rotacion["y_train"] = pd.concat(lista_y_train, ignore_index=True).squeeze()
conjuntos_alta_rotacion["y_test"] = pd.concat(lista_y_test, ignore_index=True).squeeze()

print(f"El conjunto x_train contiene {conjuntos_alta_rotacion["x_train"].shape[0]} filas y {conjuntos_alta_rotacion["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_alta_rotacion["y_train"].shape[0]} filas y {conjuntos_alta_rotacion["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_alta_rotacion["x_test"].shape[0]} filas y {conjuntos_alta_rotacion["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_alta_rotacion["y_test"].shape[0]} filas y {conjuntos_alta_rotacion["y_test"].shape[1]} columnas")

In [ ]:
# Se inicia la búsqueda aleatoria del modelo Random Forest para el conjunto alta_rotacion
search_forest_alta_rotacion: RandomizedSearchCV = RandomizedSearchCV(
    estimator=modelo_forest_search,
    param_distributions=param_search_forest,
    n_iter=N_ITER,
    verbose=1,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=SEMILLA
)

tiempo_inicial: float = time.time()
search_forest_alta_rotacion.fit(conjuntos_alta_rotacion["x_train"], conjuntos_alta_rotacion["y_train"])
tiempo_final: float = time.time()

print(
    "La búsqueda aleatoria del modelo Random Forest para el cluster alta_rotacion tardó "
    f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
)

In [ ]:
# Se muestran los valores óptimos de los hiperparámetros del algoritmo tras la búsqueda aleatoria
n_estimators_opt: int = search_forest_alta_rotacion.best_params_["n_estimators"]
max_depth_opt: int = search_forest_alta_rotacion.best_params_["max_depth"]
max_features_opt: int | str = search_forest_alta_rotacion.best_params_["max_features"]
min_samples_split_opt: int = search_forest_alta_rotacion.best_params_["min_samples_split"]
min_samples_leaf_opt: int = search_forest_alta_rotacion.best_params_["min_samples_leaf"]

i = -1
for comb in search_forest_alta_rotacion.cv_results_["params"]:
    i += 1
    if (
        comb["n_estimators"] == n_estimators_opt and 
        comb["max_depth"] == max_depth_opt and 
        comb["max_features"] == max_features_opt and
        comb["min_samples_split"] == min_samples_split_opt and
        comb["min_samples_leaf"] == min_samples_leaf_opt
    ):
        break

print(
    f"El mejor número de árboles es n_estimators={n_estimators_opt}, la profundidad máxima óptima "
    f"de los árboles es de max_depth={max_depth_opt}, el mejor criterio para partir una hoja de "
    f"un árbol según el número de variables a tener en cuenta es max_features={max_features_opt}, "
    f"el número mínimo de muestras de un split debe ser min_samples_split={min_samples_split_opt}, "
    f"y el número mínimo de muestras por hoja debe ser min_samples_leaf={min_samples_leaf_opt}", "\n\n"
)
print(
    f"Con esta combinación de hiperparámetros, durante la validación cruzada para las series temporales "
    f"se obtuvo un valor promedio del RMSE de {-search_forest_alta_rotacion.cv_results_["mean_test_score"][i]:.4f} con "
    f"una desviación estándar de {search_forest_alta_rotacion.cv_results_["std_test_score"][i]:.4f}"
)

In [ ]:
# Se entrena el modelo Random Forest con los mejores hiperparámetros empleando el cluster alta_rotacion
tiempo_inicial: float = time.time()

modelo_forest_alta_rotacion: RandomForestRegressor = RandomForestRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    max_features=max_features_opt,
    min_samples_split=min_samples_split_opt,
    min_samples_leaf=min_samples_leaf_opt,
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

modelo_forest_alta_rotacion.fit(conjuntos_alta_rotacion["x_train"], conjuntos_alta_rotacion["y_train"])

tiempo_final: float = time.time()

print(f"El modelo de alta_rotacion tardó {round(tiempo_final - tiempo_inicial, 4)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del cluster alta_rotacion
y_pred_alta_rotacion: np.ndarray = modelo_forest_alta_rotacion.predict(conjuntos_alta_rotacion["x_test"])

y_pred_alta_rotacion_df: pd.DataFrame = pd.DataFrame(
    y_pred_alta_rotacion,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in alta_rotacion_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_alta_rotacion_df: pd.DataFrame = pd.DataFrame(
    conjuntos_alta_rotacion["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_alta_rotacion: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_alta_rotacion_df, y_pred_alta_rotacion_df],
    axis=1
)

resultados_alta_rotacion["fecha"] = pd.to_datetime(resultados_alta_rotacion["fecha"])

In [ ]:
# Se calculan los promedios de los errores cometidos en el conjunto de prueba del cluster alta_rotacion
rmses: Dict[str, float] = {
    i: rmse(
        resultados_alta_rotacion[f"real_{i}"],
        resultados_alta_rotacion[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales de alta_rotacion")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster alta_rotacion_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba de alta_rotacion por producto y se exportan para la evaluación económica
rmses_alta_rotacion: List[pd.DataFrame] = []

for producto, df_producto in resultados_alta_rotacion.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_alta_rotacion.append(rmses_df)

rmses_alta_rotacion: pd.DataFrame = pd.concat(rmses_alta_rotacion, ignore_index=True)
rmses_alta_rotacion.to_csv("rmses_random_forest_alta_rotacion.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del grupo alta_rotacion
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        alta_rotacion_subset,
        resultados_alta_rotacion,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_alta_rotacion.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo alta_rotacion
rmsse_modelo_alta_rotacion: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo alta_rotacion es {rmsse_modelo_alta_rotacion:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios de alta_rotacion
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_alta_rotacion["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_alta_rotacion[resultados_alta_rotacion["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} de alta_rotacion")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se visualiza la importancia de cada atributo empleado en el modelo de alta_rotacion
plt.figure(figsize=(10, 6))
plt.bar(
    conjuntos_alta_rotacion["x_train"].columns,
    modelo_forest_alta_rotacion.feature_importances_,
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1
)

plt.title("Importancia de cada atributo del cluster alta_rotacion en las decisiones del algoritmo Random Forest")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importancia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_alta_rotacion.to_csv("res_random_forest_alta_rotacion.csv")

<a id="ej2.4"></a>
## 2.4. _Cluster estandar_

Por último, se entrena un modelo _Random Forest_ con los datos del _cluster estandar_. Este grupo es el más voluminoso de todos y recoge aquellos productos con un comportamiento muy próximo a la media de todo el conjunto de datos global. De la misma forma que con los otros modelos del _notebook_, se utiliza la función _**secuenciador()**_ para calcular el conjunto de variables de ventana móvil en el pasado, y obtener la demanda de cada producto en los distintos horizontes temporales. 

In [ ]:
# Se extrae el grupo estandar del dataset global
estandar: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "estandar"]

In [ ]:
# Se crean las nuevas variables para el cluster estandar
HORIZONTE: int = max(estandar["diasLeadtime"] + estandar["diasEntrePedidos"])
estandar_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in estandar.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    estandar_secuenciado.append(df_producto_sec)

estandar_secuenciado: pd.DataFrame = pd.concat(estandar_secuenciado, ignore_index=True)

<br><br>El modelo _Random Forest_ que se entrenará en este apartado utilizará los datos del _cluster_ de las variables calculadas con la ventana móvil cada intervalo de una semana en el pasado, junto con los atributos originales del conjunto de datos global.

In [ ]:
# Se selecciona el subconjunto de variables de interés del cluster estandar
estandar_subset: pd.DataFrame = (
    estandar_secuenciado[
        [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles", "dia_semana_str_Jueves",
            "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo"
        ] + [f"mes_str_{i}" for i in range(1, 13)] + [
            "lag_7", "lag_14", "lag_21", "lag_28", "media_rolling_7", "media_rolling_14", "media_rolling_21", "media_rolling_28",
            "max_rolling_7", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7", "std_rolling_14", "std_rolling_21",
            "std_rolling_28", "udsVenta"
        ] 
    ]
)

In [ ]:
# Matriz de correlaciones con las nuevas variables explicativas
corr: pd.DataFrame = estandar_subset.drop(columns=["producto", "fecha", "udsVenta"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlaciones del cluster estandar tras realizar un procedimiento de ingeniería de características")

plt.tight_layout()
plt.show()

La matriz de correlaciones calculada ilustra patrones muy similares a otros _clusters_ en cuanto a las variables derivadas mediante ventanas móviles, hecho que refuerza la idea de que la dependencia de la demanda respecto a su comportamiento reciente es una característica estructural común entre los distintos grupos de productos.

Finalmente, se entrena el modelo _Random Forest_ empleando los datos del subconjunto de variables del _cluster estandar_.

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster estandar
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_estandar: Dict[str, pd.DataFrame] = {}
lista_x_train: List[pd.DataFrame] = []
lista_x_test: List[pd.DataFrame] = []
lista_y_train: List[pd.DataFrame] = []
lista_y_test: List[pd.DataFrame] = []

for producto, df_producto in estandar_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes)
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    lista_x_train.append(x_producto.drop("fecha", axis=1).iloc[:PARTICION])
    lista_x_test.append(x_producto.drop("fecha", axis=1).iloc[PARTICION:])
    
    lista_y_train.append(y_producto.iloc[:PARTICION])
    lista_y_test.append(y_producto.iloc[PARTICION:])

conjuntos_estandar["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_estandar["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_estandar["y_train"] = pd.concat(lista_y_train, ignore_index=True).squeeze()
conjuntos_estandar["y_test"] = pd.concat(lista_y_test, ignore_index=True).squeeze()

print(f"El conjunto x_train contiene {conjuntos_estandar["x_train"].shape[0]} filas y {conjuntos_estandar["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_estandar["y_train"].shape[0]} filas y {conjuntos_estandar["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_estandar["x_test"].shape[0]} filas y {conjuntos_estandar["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_estandar["y_test"].shape[0]} filas y {conjuntos_estandar["y_test"].shape[1]} columnas")

In [ ]:
# Se inicia la búsqueda aleatoria del modelo Random Forest para el conjunto estandar
search_forest_estandar: RandomizedSearchCV = RandomizedSearchCV(
    estimator=modelo_forest_search,
    param_distributions=param_search_forest,
    n_iter=N_ITER,
    verbose=1,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=SEMILLA
)

tiempo_inicial: float = time.time()
search_forest_estandar.fit(conjuntos_estandar["x_train"], conjuntos_estandar["y_train"])
tiempo_final: float = time.time()

print(
    "La búsqueda aleatoria del modelo Random Forest para el cluster estandar tardó "
    f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
)

In [ ]:
# Se muestran los valores óptimos de los hiperparámetros del algoritmo tras la búsqueda aleatoria
n_estimators_opt: int = search_forest_estandar.best_params_["n_estimators"]
max_depth_opt: int = search_forest_estandar.best_params_["max_depth"]
max_features_opt: int | str = search_forest_estandar.best_params_["max_features"]
min_samples_split_opt: int = search_forest_estandar.best_params_["min_samples_split"]
min_samples_leaf_opt: int = search_forest_estandar.best_params_["min_samples_leaf"]

i = -1
for comb in search_forest_estandar.cv_results_["params"]:
    i += 1
    if (
        comb["n_estimators"] == n_estimators_opt and 
        comb["max_depth"] == max_depth_opt and 
        comb["max_features"] == max_features_opt and
        comb["min_samples_split"] == min_samples_split_opt and
        comb["min_samples_leaf"] == min_samples_leaf_opt
    ):
        break

print(
    f"El mejor número de árboles es n_estimators={n_estimators_opt}, la profundidad máxima óptima "
    f"de los árboles es de max_depth={max_depth_opt}, el mejor criterio para partir una hoja de "
    f"un árbol según el número de variables a tener en cuenta es max_features={max_features_opt}, "
    f"el número mínimo de muestras de un split debe ser min_samples_split={min_samples_split_opt}, "
    f"y el número mínimo de muestras por hoja debe ser min_samples_leaf={min_samples_leaf_opt}", "\n\n"
)
print(
    f"Con esta combinación de hiperparámetros, durante la validación cruzada para las series temporales "
    f"se obtuvo un valor promedio del RMSE de {-search_forest_estandar.cv_results_["mean_test_score"][i]:.4f} con "
    f"una desviación estándar de {search_forest_estandar.cv_results_["std_test_score"][i]:.4f}"
)

In [ ]:
# Se entrena el modelo Random Forest con los mejores hiperparámetros empleando el cluster estandar
tiempo_inicial: float = time.time()

modelo_forest_estandar: RandomForestRegressor = RandomForestRegressor(
    n_estimators=n_estimators_opt,
    max_depth=max_depth_opt,
    max_features=max_features_opt,
    min_samples_split=min_samples_split_opt,
    min_samples_leaf=min_samples_leaf_opt,
    criterion="squared_error",
    random_state=SEMILLA,
    n_jobs=-1
)

modelo_forest_estandar.fit(conjuntos_estandar["x_train"], conjuntos_estandar["y_train"])

tiempo_final: float = time.time()

print(f"El modelo del cluster estandar tardó {round(tiempo_final - tiempo_inicial, 4)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del cluster estandar
y_pred_estandar: np.ndarray = modelo_forest_estandar.predict(conjuntos_estandar["x_test"])

y_pred_estandar_df: pd.DataFrame = pd.DataFrame(
    y_pred_estandar,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in estandar_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_estandar_df: pd.DataFrame = pd.DataFrame(
    conjuntos_estandar["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_estandar: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_estandar_df, y_pred_estandar_df],
    axis=1
)

resultados_estandar["fecha"] = pd.to_datetime(resultados_estandar["fecha"])

In [ ]:
# Se calculan los promedios de los errores cometidos en el conjunto de prueba del cluster estandar
rmses: Dict[str, float] = {
    i: rmse(
        resultados_estandar[f"real_{i}"],
        resultados_estandar[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del cluster estandar")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster estandar_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster estandar por producto y se exportan para la evaluación económica
rmses_estandar: List[pd.DataFrame] = []

for producto, df_producto in resultados_estandar.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_estandar.append(rmses_df)

rmses_estandar: pd.DataFrame = pd.concat(rmses_estandar, ignore_index=True)
rmses_estandar.to_csv("rmses_random_forest_estandar.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del cluster estandar
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        estandar_subset,
        resultados_estandar,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_estandar.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo del cluster estandar
rmsse_modelo_estandar: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo del cluster estandar es {rmsse_modelo_estandar:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster estandar
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_estandar["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_estandar[resultados_estandar["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster estandar")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se visualiza la importancia de cada atributo empleado en el modelo del cluster estandar
plt.figure(figsize=(10, 6))
plt.bar(
    conjuntos_estandar["x_train"].columns,
    modelo_forest_estandar.feature_importances_,
    color="blue",
    alpha=0.6,
    edgecolor="black",
    linewidth=1
)

plt.title("Importancia de cada atributo del cluster estandar en las decisiones del algoritmo Random Forest")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importancia")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_estandar.to_csv("res_random_forest_estandar.csv")